# 05 · Inheritance & Dunders

`__repr__`, `__eq__`/`__hash__` together, inheritance and `super()`, polymorphism/duck typing, composition over inheritance, and the container dunders `__len__`/`__getitem__` etc.

### 5.1 — `__repr__` and `__str__`

`__str__` is the human-readable form (`print`, `str()`); `__repr__` is the unambiguous developer 
form (`repr()`, the REPL, containers). If you define only one, define `__repr__`.

Build `Point(x, y)` whose `repr` is `"Point(1, 2)"`.

In [2]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

# Tests
assert repr(Point(1, 2)) == "Point(1, 2)"
assert repr(Point(-3, 0)) == "Point(-3, 0)"
print("5.1 passed")

5.1 passed


### 5.2 — `__eq__` and `__hash__` together

To use a custom object as a dict key or set element, it needs `__eq__` (when are two equal?) and 
`__hash__` (an int, equal objects must hash equal). This is the same idea behind why BPE pairs are 
tuples — they're hashable and compare by value.

Build `Coord(x, y)` where two coords are equal iff x and y match, and equal coords hash equal 
(hint: hash the tuple `(x, y)`).

In [3]:
class Coord:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        if not isinstance(other, Coord):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))

# Tests
assert Coord(1, 2) == Coord(1, 2)
assert Coord(1, 2) != Coord(3, 4)
s = {Coord(1, 2), Coord(1, 2), Coord(3, 4)}
assert len(s) == 2                       # equal coords dedupe
d = {Coord(0, 0): "origin"}
assert d[Coord(0, 0)] == "origin"        # lookup by an equal-but-different object
print("5.2 passed")

5.2 passed


### 5.3 — Inheritance and `super()`

A subclass inherits from a parent and can extend it. `super().__init__(...)` calls the parent's 
constructor so you don't repeat its setup.

`Animal` stores a `name` and has `speak()` returning `"..."`. Build `Dog(Animal)` that calls 
`super().__init__`, and overrides `speak()` to return `"Woof"`.

In [4]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

class Dog(Animal):
    def __init__(self, name):
        super().__init__(name)

    def speak(self):
        return "Woof"

# Tests
d = Dog("Rex")
assert d.name == "Rex"          # inherited setup
assert d.speak() == "Woof"      # overridden
assert Animal("x").speak() == "..."
assert isinstance(d, Animal)    # a Dog IS an Animal
print("5.3 passed")

5.3 passed


### 5.4 — Polymorphism / duck typing

"If it walks like a duck..." — Python doesn't care about an object's type, only that it has the 
method you call. Write `make_all_speak(animals)` that returns a list of `.speak()` results for any 
list of objects that have a `speak` method — regardless of their class.

```
make_all_speak([Dog("a"), Cat("b")]) → ["Woof", "Meow"]
```

In [5]:
class Cat:
    def speak(self):
        return "Meow"

def make_all_speak(animals):
    return [s.speak() for s in animals]

# Tests
class Duck:
    def speak(self):
        return "Quack"
assert make_all_speak([Cat(), Duck()]) == ["Meow", "Quack"]
print("5.4 passed")

5.4 passed


### 5.5 — Composition over inheritance

Often "has-a" beats "is-a". A `Car` *has an* `Engine` rather than *being* one. This keeps pieces 
swappable — directly relevant to how you'll make Mara's components modular.

Build `Engine` with a `start()` returning `"vroom"`, and `Car` that holds an `Engine` instance and 
exposes `start()` by delegating to its engine.

In [7]:
class Engine:
    def start(self):
        return "vroom"

class Car:
    def __init__(self):
        self.engine = Engine()

    def start(self):
        return self.engine.start()


# Tests
c = Car()
assert c.start() == "vroom"
assert isinstance(c.engine, Engine)
print("5.5 passed")

5.5 passed


### 5.6 — `__len__` and `__getitem__`

Implementing dunders makes your object behave like a built-in. `__len__` powers `len()`; 
`__getitem__` powers indexing `obj[i]` (and, for free, iteration).

Build a `Playlist` wrapping a list of songs, supporting `len(playlist)` and `playlist[i]`.

In [8]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    def __getitem__(self, i):
        return self.songs[i]

# Tests
p = Playlist(["a", "b", "c"])
assert len(p) == 3
assert p[0] == "a"
assert p[-1] == "c"
assert [s for s in p] == ["a", "b", "c"]   # iteration works via __getitem__
print("5.6 passed")

5.6 passed
